# 01 - Data Cleaning
**Netflix Data Analysis Portfolio Project**

This notebook loads the raw synthetic Netflix titles catalog, profiles its data-quality issues, and walks through every cleaning decision step by step. The same logic lives in `src/data_cleaning.py` as a reusable, tested pipeline; this notebook exists to show the reasoning and diagnostics behind each step.

In [1]:
import sys
sys.path.append('../src')
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid')
%matplotlib inline


## 1. Load Raw Data

In [2]:
df = pd.read_csv('../dataset/netflix_titles.csv')
print(df.shape)
df.head()

(8819, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Scarlet Diaries 974,Hana Kobayashi,"Jordan Blake, Samuel Osei, Nina Kowalski, Chlo...",South Korea,"September 20, 2020",2015,TV-14,102 min,Thrillers,"In a world on the edge of collapse, Antonio Si..."
1,s2,Movie,Sacred Symphony 936,Jin-ho Park,"Omar Haddad, Kwame Boateng, Lily Zhou, Felix N...","Germany, Japan","July 14, 2021",2020,TV-MA,54 min,"Independent Movies, Comedies","In a world on the edge of collapse, Omar Hadda..."
2,s3,TV Show,Secret Republic Squad,NaN,Adam Fischer,Canada,"November 9, 2018",2017,TV-PG,1 Season,Anime Series,"When Ethan Cole uncovers a dangerous secret, E..."
3,s4,TV Show,Endless Lantern: The Next Chapter,NaN,Samuel Osei,United Kingdom,"July 26, 2020",2014,TV-G,4 Seasons,"Crime TV Shows, TV Comedies","In a world on the edge of collapse, Adam Fisch..."
4,s5,Movie,Scarlet Season 110,Priya Nair,"Kwame Boateng, Jordan Blake",United States,"January 16, 2021",2021,TV-G,96 min,"Dramas, Action & Adventure","After a chance encounter, Maya Chen and Sofia ..."


## 2. Data Understanding
### 2.1 Shape, Dtypes, Memory

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8819 entries, 0 to 8818
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8819 non-null   str  
 1   type          8819 non-null   str  
 2   title         8819 non-null   str  
 3   director      5382 non-null   str  
 4   cast          8038 non-null   str  
 5   country       7988 non-null   str  
 6   date_added    8819 non-null   str  
 7   release_year  8819 non-null   int64
 8   rating        8802 non-null   str  
 9   duration      8819 non-null   str  
 10  listed_in     8819 non-null   str  
 11  description   8819 non-null   str  
dtypes: int64(1), str(11)
memory usage: 826.9 KB


In [4]:
print('Memory usage (MB):', round(df.memory_usage(deep=True).sum() / 1e6, 2))

Memory usage (MB): 6.8


### 2.2 Summary Statistics

In [5]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
show_id,8819,8819,s1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,8819,2,Movie,6187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,8819,8789,Northern Circuit 279,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
director,5382,20,Marcus Whitfield,566,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cast,8038,6018,Yuki Tanaka,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,7988,765,United States,1974,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_added,8819,2003,"July 10, 2021",20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_year,8819.0,NaN,NaN,NaN,2014.789092,10.714282,1945.0,2015.0,2017.0,2019.0,2021.0
rating,8802,16,TV-MA,2821,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration,8819,142,1 Season,1923,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2.3 Missing Values

In [6]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

,missing_count,missing_pct
director,3437,39.0
country,831,9.4
cast,781,8.9
rating,17,0.2
title,0,0.0
type,0,0.0
show_id,0,0.0
date_added,0,0.0
release_year,0,0.0
duration,0,0.0


**Observation:** `director` is missing for a large share of records -- this mirrors the real Netflix catalog, where TV show directors are frequently uncredited or omitted, while movies are more consistently credited. `cast` and `country` have smaller but still meaningful gaps.

### 2.4 Duplicate Records

In [7]:
exact_dupes = df.duplicated(subset=['title', 'type', 'director', 'date_added', 'release_year']).sum()
print('Exact duplicate rows:', exact_dupes)

Exact duplicate rows: 12


### 2.5 Data Quality Issue: Invalid Rating Values
A known real-world Netflix data issue is a column shift where a duration string ends up in the `rating` field. We check for this explicitly.

In [8]:
VALID_RATINGS = {'TV-MA','TV-14','TV-PG','TV-Y7','TV-Y','TV-G','R','PG-13','PG','G','NC-17','NR','UR'}
bad_ratings = df[~df['rating'].isin(VALID_RATINGS) & df['rating'].notna()]
print('Rows with invalid rating values:', len(bad_ratings))
bad_ratings[['title', 'rating', 'duration']].head()

Rows with invalid rating values: 6


,title,rating,duration
128,Scarlet Chronicles 880,107 min,107 min
711,Forgotten Mirror: Origins,1 Season,1 Season
2554,Secret Code: Origins,1 Season,1 Season
4224,Midnight Uprising Files,1 Season,1 Season
7886,Silent Circuit 881,104 min,104 min


## 3. Data Cleaning
Applying the full pipeline from `src/data_cleaning.py`. Each step is documented with the *why*, not just the *what*, inside that module's docstrings.

In [9]:
from data_cleaning import run_pipeline
clean_df = run_pipeline()
clean_df.shape

Loaded 8819 raw rows.
Dropped 12 exact duplicate rows.
Fixed 6 rows where rating held a non-standard value.
Dropped 0 rows with unparseable date_added.


All validation checks passed.


Saved cleaned dataset -> /home/claude/Netflix-Data-Analysis/dataset/netflix_titles_clean.csv (8807 rows, 24 columns)


(8807, 24)

## 4. Post-Cleaning Validation

In [10]:
print('Remaining nulls:\n', clean_df.isna().sum()[clean_df.isna().sum() > 0])
print('\nUnique content types:', clean_df['type'].unique())
print('Rating values now all valid:', clean_df['rating'].isin(VALID_RATINGS).all())

Remaining nulls:
 duration_minutes    2629
duration_seasons    6178
dtype: int64

Unique content types: <StringArray>
['Movie', 'TV Show']
Length: 2, dtype: str
Rating values now all valid: True


In [11]:
clean_df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added,month_name_added,duration_minutes,duration_seasons,primary_genre,primary_country,num_genres,num_cast,content_age_years,is_international,decade_released
0,s1,Movie,Scarlet Diaries 974,Hana Kobayashi,"Jordan Blake, Samuel Osei, Nina Kowalski, Chlo...",South Korea,2020-09-20,2015,TV-14,102 min,Thrillers,"In a world on the edge of collapse, Antonio Si...",2020,9,September,102.0,NaN,Thrillers,South Korea,1,4,5,True,2010
1,s2,Movie,Sacred Symphony 936,Jin-ho Park,"Omar Haddad, Kwame Boateng, Lily Zhou, Felix N...","Germany, Japan",2021-07-14,2020,TV-MA,54 min,"Independent Movies, Comedies","In a world on the edge of collapse, Omar Hadda...",2021,7,July,54.0,NaN,Independent Movies,Germany,2,5,1,True,2020
2,s3,TV Show,Secret Republic Squad,Unknown,Adam Fischer,Canada,2018-11-09,2017,TV-PG,1 Season,Anime Series,"When Ethan Cole uncovers a dangerous secret, E...",2018,11,November,NaN,1.0,Anime Series,Canada,1,1,1,True,2010
3,s4,TV Show,Endless Lantern: The Next Chapter,Unknown,Samuel Osei,United Kingdom,2020-07-26,2014,TV-G,4 Seasons,"Crime TV Shows, TV Comedies","In a world on the edge of collapse, Adam Fisch...",2020,7,July,NaN,4.0,Crime TV Shows,United Kingdom,2,1,6,True,2010
4,s5,Movie,Scarlet Season 110,Priya Nair,"Kwame Boateng, Jordan Blake",United States,2021-01-16,2021,TV-G,96 min,"Dramas, Action & Adventure","After a chance encounter, Maya Chen and Sofia ...",2021,1,January,96.0,NaN,Dramas,United States,2,2,0,False,2020


## 5. Summary of Cleaning Decisions

| Issue | Decision | Rationale |
|---|---|---|
| 12 exact duplicate rows | Dropped | No new information, would double-count in every aggregation |
| Missing `director`/`cast`/`country` | Filled with `'Unknown'` | Preserves row for volume analyses; kept explicit rather than silently dropped |
| Missing `rating` | Filled with mode rating **within content type** | Movie and TV rating scales differ, so a global mode would be wrong |
| Invalid rating values (duration text) | Nulled out, then imputed | Known real-world column-shift defect |
| `date_added` as text | Parsed to `datetime64` | Enables year/month feature engineering and time-series analysis |
| Inconsistent whitespace/casing in `country` | Trimmed + title-cased | Prevents `'united states'` and `'United States'` fragmenting into separate groups |
| Flat categorical columns (`listed_in`, `country`, `cast`) | Added `primary_genre` / `primary_country` / `num_cast` / `num_genres` | Enables clean single-value groupbys while still preserving full multi-value columns |
| `duration` mixed units (min vs. seasons) | Split into `duration_minutes` / `duration_seasons` | Makes both fields directly usable as numeric features |

The cleaned dataset is saved to `dataset/netflix_titles_clean.csv` and is the single source of truth for every subsequent notebook and script in this project.